In [0]:
import pyspark.sql.functions as F

#1. Load bronze.trips table

In [0]:
df = spark.table('warsaw_gtfs.bronze.trips')
df.display()

In [0]:
# data issues
#1.Unfriendly wheelchair_accessible values
#2.Unfriendly direction id values
#3.Unfriendly variant_code values.

#2. Transformations

##Give more friendly values to wheelchair_accessible column

In [0]:
df1 = df.withColumn(
    'wheelchair_accessible',
    F.when(F.col('wheelchair_accessible') == 0, 'no info')
    .when(F.col('wheelchair_accessible') == 1, 'yes')
    .otherwise('no')
    )

df2 = df1.withColumn(
    'direction_id',
    F.when(F.col('direction_id') == 0, 'outbound')
    .otherwise('inbound')
    )


##Give more friendly values to variant_code

In [0]:
# count variant_types by trip to identify the main variant type and shortened/exception variants
df_trip_types = df2.groupBy('trip_headsign', 'variant_code').agg(F.count('variant_code').alias('variant_code_count'))

df_trip_types.filter(F.col('trip_headsign') == 'Kabaty').display()


In [0]:

# label variant_codes appropriately
df_trip_types = df_trip_types.withColumn('variant_label', 
                         F.when(F.col('variant_code_count') > 100, 'Main')
                         .otherwise('shortened/course change')
                         )
# join to original df
final_df = df2.join(df_trip_types, 
           on = ['trip_headsign', 'variant_code'],
           how = 'left')

final_df.display()

In [0]:
# drop unnessecary columns and rename variant_type
final_df = final_df.drop('variant_code_count', 'variant_code').withColumnRenamed('variant_label', 'trip_variant')
# handle nulls
final_df = final_df.na.fill('n/a', subset=['trip_short_name', 'trip_variant', 'block_short_name', 'fleet_type', 'block_id'])
final_df.filter(F.col("trip_headsign") == 'Bemowo').display()

#3. Write data to silver layer 

In [0]:
final_df.write.mode('overwrite').format('delta').saveAsTable('warsaw_gtfs.silver.trips')